# Dense vectors (embeddings) in pytorch

The TODO is not to be turned in, it's just to learn how the embeddings layer works in pytorch.



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

/home/fatoumata/Documents/NLP/TP2/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Embedding layer : torch.nn.Embeddings

We've seen that the forward propagation within a neural network may use a "lookup" operation: given an word id (or the id of any other kind of symbol), retrieve a corresponding vector.

This can be handled in pytorch using an instance of the torch.nn.Embeddings class:

https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html

The torch.nn.Embeddings class acts as an "embedding layer" within a neural network, namely it simply corresponds to a matrix of vectors (or "embeddings") for each symbol id in a vocabulary of symbols.

The forward propagation for such a layer corresponds to a lookup operation into a matrix of embeddings.

For instance,
- given an embedding layer E containing embeddings of size 100
- given X a 2x3 matrix containing word ids (imagine two rows, each containing 3 word ids),
- performing the forward operation of E on X will give a 2x3x100 tensor

More generally, **A nn.Embeddings instance, with embeddings of size h, can transform any tensor containing ids into a tensor of vectors for these ids**: if the input shape is (d1, d2, ... dn), the output shape will be (d1, d2, ..., dn, h).

**Moreover, being a subclass of the nn.Module class, nn.Embedding will handle backpropagation : the embeddings can be considered as parameters of a neural network**

### Example with random initialization

In [2]:
# see for instance https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html

VOCAB_SIZE = 10
EMBEDDING_SIZE = 5
my_embedding_layer = nn.Embedding(VOCAB_SIZE, EMBEDDING_SIZE)

# one way among others to randomly initialize the embeddings
initrange = 0.5 / EMBEDDING_SIZE
my_embedding_layer.weight.data.uniform_(-initrange, initrange)
print("10 embeddings of length 5:\n", my_embedding_layer.weight.data)


10 embeddings of length 5:
 tensor([[-0.0135, -0.0853,  0.0220, -0.0313, -0.0393],
        [-0.0773, -0.0401, -0.0579,  0.0233, -0.0808],
        [-0.0107, -0.0350, -0.0404,  0.0775,  0.0124],
        [ 0.0309,  0.0058,  0.0528, -0.0789,  0.0214],
        [-0.0604, -0.0236, -0.0351,  0.0245,  0.0006],
        [ 0.0302, -0.0426,  0.0032, -0.0049, -0.0242],
        [ 0.0731,  0.0992,  0.0153,  0.0777, -0.0132],
        [-0.0154, -0.0667,  0.0998, -0.0213,  0.0543],
        [ 0.0387, -0.0133,  0.0735,  0.0391,  0.0551],
        [-0.0813, -0.0124, -0.0564,  0.0859,  0.0247]])


### Initializing with pre-trained embeddings

See the `torch.nn.Embeddings.from_pretrained` method.

Crucially, its `freeze` option will control **whether or not to consider the embeddings as parameters**, i.e. whether or not to update the embeddings in later learning process.


###  Example for an input tensor with a single axis

In [3]:
# Creation of a vector of ids
# NB: we cannot simply use rand, cf. this gives real values between 0 and 1
a = torch.rand(4)
print(a)

# we rather need random integers to play the role of symbol ids (e.g. word ids)
wids = torch.LongTensor(4).random_(0, VOCAB_SIZE-1)
print("wids:", wids)
print(wids.size())

tensor([0.6906, 0.2426, 0.4137, 0.2351])
wids: tensor([3, 1, 4, 7])
torch.Size([4])


In [4]:
# transformation into embeddings
# reminder: for any subclass of nn.Module (hence for nn.Embedding)
#           the forward propagation is done by calling the intance name
wembs = my_embedding_layer(wids)
print("Shape after transformation into embeddings:", wembs.size())
# the embedding of wids[0], i.e. the embedding of the word whose id is 7
print("The embedding for the id %d : "% wids[0], wembs[0])


Shape after transformation into embeddings: torch.Size([4, 5])
The embedding for the id 3 :  tensor([ 0.0309,  0.0058,  0.0528, -0.0789,  0.0214],
       grad_fn=<SelectBackward0>)


###  For an input matrix (two axes)

In [5]:
mat = torch.LongTensor(3,2).random_(0, VOCAB_SIZE-1)
print(mat)
print(mat.size())

tensor([[6, 1],
        [2, 6],
        [2, 1]])
torch.Size([3, 2])


In [6]:
# idem, the embedding layer transforms all the ids into dense vectors
# => we obtain a tensor with 3 axes
emb_mat = my_embedding_layer(mat)
print(emb_mat.size())
# the embedding for the id at mat[0,1]
print(mat[0,1])
print(emb_mat[0,1,:])
print(emb_mat)

torch.Size([3, 2, 5])
tensor(1)
tensor([-0.0773, -0.0401, -0.0579,  0.0233, -0.0808],
       grad_fn=<SelectBackward0>)
tensor([[[ 0.0731,  0.0992,  0.0153,  0.0777, -0.0132],
         [-0.0773, -0.0401, -0.0579,  0.0233, -0.0808]],

        [[-0.0107, -0.0350, -0.0404,  0.0775,  0.0124],
         [ 0.0731,  0.0992,  0.0153,  0.0777, -0.0132]],

        [[-0.0107, -0.0350, -0.0404,  0.0775,  0.0124],
         [-0.0773, -0.0401, -0.0579,  0.0233, -0.0808]]],
       grad_fn=<EmbeddingBackward0>)


### TODO: concatenation and sum of embeddings
Suppose we consider that the `mat` tensor above corresponds to a batch of word id sequences. Namely here that it corresponds to:
- 3 input examples
- each example containing 2 word ids


**1. Use the `view` method to print a tensor containing for each example the concatenation of the 2 word embeddings
(what is its shape?)**



In [14]:
#concatenation of the 2 first embeddings of the first line
concat_emb = []
concat_emb.append(torch.cat((emb_mat[0,0,:], emb_mat[0,1,:]), dim=0))
concat_emb.append(torch.cat((emb_mat[1,0,:], emb_mat[1,1,:]), dim=0))
concat_emb.append(torch.cat((emb_mat[2,0,:], emb_mat[2,1,:]), dim=0))
concat_emb = torch.stack(concat_emb)
print(concat_emb.size())
print(concat_emb)

torch.Size([3, 10])
tensor([[ 0.0731,  0.0992,  0.0153,  0.0777, -0.0132, -0.0773, -0.0401, -0.0579,
          0.0233, -0.0808],
        [-0.0107, -0.0350, -0.0404,  0.0775,  0.0124,  0.0731,  0.0992,  0.0153,
          0.0777, -0.0132],
        [-0.0107, -0.0350, -0.0404,  0.0775,  0.0124, -0.0773, -0.0401, -0.0579,
          0.0233, -0.0808]], grad_fn=<StackBackward0>)


**2. Print a tensor containing, for each example, the sum of the 2 word embeddings
(what is its shape?)**


In [ ]:
# here is what the first row of the resulting tensor should look like
print(emb_mat[0,0,:] + emb_mat[0,1,:])

tensor([-0.0042,  0.0591, -0.0426,  0.1010, -0.0940], grad_fn=<AddBackward0>)
torch.Size([3, 5])
tensor([[-0.0042,  0.0591, -0.0426,  0.1010, -0.0940],
        [ 0.0624,  0.0642, -0.0251,  0.1551, -0.0008],
        [-0.0880, -0.0751, -0.0983,  0.1008, -0.0684]],
       grad_fn=<StackBackward0>)


In [ ]:

# TODO
#sum of the 2 first embeddings of the first line
sum_emb = []
sum_emb.append(emb_mat[0,0,:] + emb_mat[0,1,:])
sum_emb.append(emb_mat[1,0,:] + emb_mat[1,1,:])
sum_emb.append(emb_mat[2,0,:] + emb_mat[2,1,:])
sum_emb = torch.stack(sum_emb)
print(sum_emb.size())
print(sum_emb)